In [2]:
# ==============================================================================
# COLAB — CONSOLIDADOR E PADRONIZADOR DOS RESULTADOS DO BENCHMARK
# ==============================================================================
# Fluxo:
# 1. Solicita upload de arquivos .zip ou .csv
# 2. Extrai automaticamente os ZIPs
# 3. Procura todos os CSVs
# 4. Padroniza nomes dos arquivos
# 5. Cria índice consolidado
# 6. Compacta a saída final
# 7. Faz download automático do ZIP final
# ==============================================================================

import os
import re
import shutil
import zipfile
import unicodedata
from pathlib import Path

import pandas as pd

from google.colab import files


# ==============================================================================
# CONFIGURAÇÕES
# ==============================================================================

WORK_DIR = Path("/content/benchmark_consolidation")
INPUT_DIR = WORK_DIR / "input"
EXTRACT_DIR = WORK_DIR / "extracted"
DEST_DIR = WORK_DIR / "RESULTADOS_CONSOLIDADOS_FINAIS"

ZIP_OUTPUT_NAME = "RESULTADOS_CONSOLIDADOS_FINAIS.zip"

FILE_EXTENSION = ".csv"

CREATE_ZIP = True


# ==============================================================================
# LIMPEZA DO AMBIENTE
# ==============================================================================

def reset_workspace():
    if WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)

    INPUT_DIR.mkdir(parents=True, exist_ok=True)
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    DEST_DIR.mkdir(parents=True, exist_ok=True)

    if Path(ZIP_OUTPUT_NAME).exists():
        Path(ZIP_OUTPUT_NAME).unlink()


# ==============================================================================
# FUNÇÕES AUXILIARES
# ==============================================================================

def normalize_text(text: str) -> str:
    text = str(text)
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))
    text = text.lower()
    text = text.replace("-", "_")
    text = text.replace(" ", "_")
    text = re.sub(r"_+", "_", text)
    return text


def safe_filename(text: str) -> str:
    text = str(text).strip()
    text = re.sub(r"\s+", "_", text)
    text = re.sub(r"[^A-Za-z0-9_\-.]+", "", text)
    text = re.sub(r"_+", "_", text)
    return text


def detect_attack(path: Path) -> str:
    s = normalize_text(path.as_posix())

    if (
        "cumulativo" in s
        or "cumulative" in s
        or "asr_fixo" in s
        or "asrfixo" in s
        or "slowdrift" in s
        or "slow_drift" in s
        or "drift" in s
    ):
        return "Cumulative"

    if (
        "modelreplacement" in s
        or "model_replacement" in s
        or "scaling" in s
        or "escala" in s
        or "escalonamento" in s
    ):
        return "Scaling"

    if (
        "sybil" in s
        or "identityflood" in s
        or "identity_flood" in s
    ):
        return "Sybil"

    if (
        "onoff" in s
        or "on_off" in s
        or "on-off" in path.as_posix().lower()
        or "sleepingcell" in s
        or "sleeping_cell" in s
        or "sleeping" in s
    ):
        return "OnOff"

    return "UnknownAttack"


def detect_dataset(path: Path) -> str:
    original = path.as_posix()
    s = normalize_text(original)

    if (
        "dataset_a" in s
        or "cps_netflow" in s
        or "cps-netflow" in original.lower()
    ):
        return "A_CPS-NetFlow-IDS"

    if (
        "dataset_b" in s
        or "packet_level" in s
        or "packet-level" in original.lower()
        or "botnet" in s
    ):
        return "B_Packet-Level-Botnet-Set"

    if (
        "dataset_c" in s
        or "mqtt_iot" in s
        or "mqtt-iot" in original.lower()
        or "mqtt" in s
    ):
        return "C_MQTT-IoT-Flood"

    if (
        "dataset_d" in s
        or "aggregated_traffic" in s
        or "aggregated traffic" in original.lower()
    ):
        return "D_Aggregated-Traffic-Set"

    return "UnknownDataset"


def detect_file_type(path: Path) -> str:
    name = normalize_text(path.name)

    if (
        "resultados_brutos" in name
        or "raw_results" in name
        or name == "raw_results.csv"
    ):
        return "raw_results"

    if (
        "summary_by_stage" in name
        or "resumo_estatisticas" in name
        or "summary_stats" in name
        or "stagewise" in name
    ):
        return "summary_by_stage"

    if (
        "summary_by_strategy" in name
        or "resumo_por_estrategia" in name
        or "por_estrategia" in name
    ):
        return "summary_by_strategy"

    if (
        "final_stage_ranking" in name
        or "ranking" in name
    ):
        return "final_stage_ranking"

    if (
        "execution_report" in name
        or "relatorio_execucao" in name
        or "relatorio" in name
    ):
        return "execution_report"

    if (
        "diagnostic" in name
        or "diagnostico" in name
    ):
        return "diagnostic"

    if (
        "all_raw_results" in name
    ):
        return "all_raw_results"

    if (
        "all_summary_by_stage" in name
    ):
        return "all_summary_by_stage"

    if (
        "all_final_stage_ranking" in name
    ):
        return "all_final_stage_ranking"

    if (
        "all_diagnostic" in name
    ):
        return "all_diagnostic"

    if (
        "consolidated" in name
        or "consolidado" in name
    ):
        return "global_consolidated"

    return "other_csv"


def detect_scope(path: Path) -> str:
    name = normalize_text(path.name)

    if name.startswith("all_") or "all_" in name:
        return "global"

    return "attack_dataset"


def build_standard_filename(path: Path) -> str:
    attack = detect_attack(path)
    dataset = detect_dataset(path)
    file_type = detect_file_type(path)
    scope = detect_scope(path)

    if scope == "global":
        new_name = f"GLOBAL_{file_type}.csv"
    else:
        new_name = f"{attack}_{dataset}_{file_type}.csv"

    return safe_filename(new_name)


def avoid_overwrite(dest_path: Path) -> Path:
    if not dest_path.exists():
        return dest_path

    stem = dest_path.stem
    suffix = dest_path.suffix
    parent = dest_path.parent

    counter = 2

    while True:
        candidate = parent / f"{stem}_v{counter}{suffix}"
        if not candidate.exists():
            return candidate
        counter += 1


def create_zip_from_folder(folder_path: Path, zip_path: Path):
    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
        for file_path in folder_path.rglob("*"):
            if file_path.is_file():
                arcname = file_path.relative_to(folder_path)
                zipf.write(file_path, arcname=arcname)


# ==============================================================================
# UPLOAD E EXTRAÇÃO
# ==============================================================================

def upload_and_extract_files():
    print("=" * 90)
    print("UPLOAD DOS RESULTADOS DO BENCHMARK")
    print("=" * 90)
    print("Faça upload de arquivos .zip ou .csv.")
    print("Você pode enviar vários arquivos de uma vez.")
    print("=" * 90)

    uploaded = files.upload()

    if not uploaded:
        raise RuntimeError("Nenhum arquivo enviado.")

    uploaded_files = []

    for filename, content in uploaded.items():
        input_path = INPUT_DIR / filename

        with open(input_path, "wb") as f:
            f.write(content)

        uploaded_files.append(input_path)

    print("\nArquivos enviados:")
    for p in uploaded_files:
        print(f" - {p.name}")

    print("\nExtraindo arquivos ZIP...")

    for p in uploaded_files:
        if p.suffix.lower() == ".zip":
            extract_target = EXTRACT_DIR / p.stem
            extract_target.mkdir(parents=True, exist_ok=True)

            try:
                with zipfile.ZipFile(p, "r") as z:
                    z.extractall(extract_target)

                print(f"Extraído: {p.name} -> {extract_target}")

            except Exception as e:
                print(f"Erro ao extrair {p.name}: {e}")

        elif p.suffix.lower() == ".csv":
            # Copia CSV direto para a pasta extracted/direct_csv
            direct_dir = EXTRACT_DIR / "direct_csv"
            direct_dir.mkdir(parents=True, exist_ok=True)
            shutil.copy2(p, direct_dir / p.name)

    print("\nUpload e extração concluídos.")


# ==============================================================================
# CONSOLIDAÇÃO
# ==============================================================================

def consolidate_results():
    source_path = EXTRACT_DIR
    dest_path = DEST_DIR

    print("\n" + "=" * 90)
    print("PADRONIZAÇÃO DOS RESULTADOS DO BENCHMARK")
    print("=" * 90)
    print(f"Origem:  {source_path}")
    print(f"Destino: {dest_path}")
    print(f"Filtro:  *{FILE_EXTENSION}")
    print("-" * 90)

    all_files = list(source_path.rglob(f"*{FILE_EXTENSION}"))

    print(f"Arquivos CSV encontrados: {len(all_files)}")
    print("-" * 90)

    if not all_files:
        raise RuntimeError("Nenhum CSV encontrado nos arquivos enviados.")

    records = []
    copied_count = 0
    skipped_count = 0

    for file_path in all_files:
        try:
            attack = detect_attack(file_path)
            dataset = detect_dataset(file_path)
            file_type = detect_file_type(file_path)
            scope = detect_scope(file_path)

            new_name = build_standard_filename(file_path)

            final_dest_path = dest_path / new_name
            final_dest_path = avoid_overwrite(final_dest_path)

            shutil.copy2(file_path, final_dest_path)

            records.append({
                "original_path": file_path.as_posix(),
                "original_name": file_path.name,
                "new_name": final_dest_path.name,
                "destination_path": final_dest_path.as_posix(),
                "scope": scope,
                "attack": attack,
                "dataset": dataset,
                "file_type": file_type,
                "size_bytes": file_path.stat().st_size
            })

            print(f"Copiado: {final_dest_path.name}")
            copied_count += 1

        except Exception as e:
            skipped_count += 1
            print(f"Erro ao processar {file_path}: {e}")

    index_df = pd.DataFrame(records)

    if not index_df.empty:
        index_path = dest_path / "index_resultados_consolidados.csv"
        index_df.to_csv(index_path, index=False)

        print("-" * 90)
        print("Resumo por tipo de arquivo:")
        print(index_df["file_type"].value_counts().to_string())

        print("-" * 90)
        print("Resumo por ataque:")
        print(index_df["attack"].value_counts().to_string())

        print("-" * 90)
        print("Resumo por dataset:")
        print(index_df["dataset"].value_counts().to_string())

        print("-" * 90)
        print(f"Índice criado: {index_path}")

    print("-" * 90)
    print("Consolidação concluída.")
    print(f"Arquivos copiados: {copied_count}")
    print(f"Arquivos com erro: {skipped_count}")

    return copied_count, skipped_count


# ==============================================================================
# MAIN
# ==============================================================================

def main():
    reset_workspace()

    upload_and_extract_files()

    copied_count, skipped_count = consolidate_results()

    if copied_count == 0:
        raise RuntimeError("Nenhum arquivo foi consolidado. Verifique os arquivos enviados.")

    zip_path = Path("/content") / ZIP_OUTPUT_NAME

    create_zip_from_folder(DEST_DIR, zip_path)

    print("\n" + "=" * 90)
    print("ARQUIVO FINAL GERADO")
    print("=" * 90)
    print(f"ZIP: {zip_path}")
    print("=" * 90)

    files.download(zip_path.as_posix())


if __name__ == "__main__":
    main()

UPLOAD DOS RESULTADOS DO BENCHMARK
Faça upload de arquivos .zip ou .csv.
Você pode enviar vários arquivos de uma vez.


Saving unified_results (1).zip to unified_results (1).zip

Arquivos enviados:
 - unified_results (1).zip

Extraindo arquivos ZIP...
Extraído: unified_results (1).zip -> /content/benchmark_consolidation/extracted/unified_results (1)

Upload e extração concluídos.

PADRONIZAÇÃO DOS RESULTADOS DO BENCHMARK
Origem:  /content/benchmark_consolidation/extracted
Destino: /content/benchmark_consolidation/RESULTADOS_CONSOLIDADOS_FINAIS
Filtro:  *.csv
------------------------------------------------------------------------------------------
Arquivos CSV encontrados: 84
------------------------------------------------------------------------------------------
Copiado: GLOBAL_summary_by_stage.csv
Copiado: GLOBAL_diagnostic.csv
Copiado: GLOBAL_raw_results.csv
Copiado: GLOBAL_final_stage_ranking.csv
Copiado: OnOff_C_MQTT-IoT-Flood_final_stage_ranking.csv
Copiado: OnOff_C_MQTT-IoT-Flood_summary_by_stage.csv
Copiado: OnOff_C_MQTT-IoT-Flood_summary_by_strategy.csv
Copiado: OnOff_C_MQTT-IoT-Flood_executi

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
# ==============================================================================
# COLAB — GERADOR PADRONIZADO DE GRÁFICOS STAGE-WISE
# ==============================================================================
# Fluxo:
# 1. Solicita upload de arquivos .zip ou .csv
# 2. Extrai automaticamente os ZIPs
# 3. Localiza arquivos brutos:
#       - raw_results.csv
#       - Resultados_Brutos.csv
#       - Resultados_Brutos_Scaling.csv
#       - Resultados_Brutos_Sybil.csv
#       - Raw_Results_OnOff.csv
# 4. Ignora resumos, relatórios, rankings e arquivos consolidados
# 5. Padroniza ataques: Cumulative, Scaling, Sybil, OnOff
# 6. Padroniza datasets: A, B, C, D
# 7. Gera gráficos:
#       - F1_over_stage.png
#       - ASR_over_stage.png
#       - Time_over_stage.png
# 8. Gera CSVs consolidados
# 9. Compacta e faz download automático
# ==============================================================================

import re
import zipfile
import shutil
import unicodedata
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from google.colab import files


# ==============================================================================
# CONFIGURAÇÕES
# ==============================================================================

WORK_DIR = Path("/content/stagewise_graphs")
INPUT_DIR = WORK_DIR / "input"
EXTRACT_DIR = WORK_DIR / "extracted"
OUT_DIR = WORK_DIR / "figs_by_dataset"

ZIP_OUTPUT_NAME = "figs_stagewise_by_dataset_ALL_attacks_FIXED_FULL.zip"

GENERATE_TIME_PLOTS = True


# ==============================================================================
# LIMPEZA DO AMBIENTE
# ==============================================================================

def reset_workspace():
    if WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)

    INPUT_DIR.mkdir(parents=True, exist_ok=True)
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    zip_path = Path("/content") / ZIP_OUTPUT_NAME
    if zip_path.exists():
        zip_path.unlink()

    for fname in [
        "benchmark_all_raw_consolidated.csv",
        "benchmark_stagewise_stats.csv",
        "benchmark_final_stage_ranking.csv",
        "benchmark_diagnostic_by_attack_dataset.csv"
    ]:
        p = Path("/content") / fname
        if p.exists():
            p.unlink()


# ==============================================================================
# UPLOAD E EXTRAÇÃO
# ==============================================================================

def upload_and_extract_files():
    print("=" * 90)
    print("UPLOAD DOS RESULTADOS DO BENCHMARK")
    print("=" * 90)
    print("Faça upload de arquivos .zip ou .csv.")
    print("Você pode enviar vários arquivos de uma vez.")
    print("=" * 90)

    uploaded = files.upload()

    if not uploaded:
        raise RuntimeError("Nenhum arquivo enviado.")

    uploaded_files = []

    for filename, content in uploaded.items():
        input_path = INPUT_DIR / filename

        with open(input_path, "wb") as f:
            f.write(content)

        uploaded_files.append(input_path)

    print("\nArquivos enviados:")
    for p in uploaded_files:
        print(f" - {p.name}")

    print("\nExtraindo arquivos ZIP...")

    for p in uploaded_files:
        if p.suffix.lower() == ".zip":
            extract_target = EXTRACT_DIR / p.stem
            extract_target.mkdir(parents=True, exist_ok=True)

            try:
                with zipfile.ZipFile(p, "r") as z:
                    z.extractall(extract_target)

                print(f"Extraído: {p.name} -> {extract_target}")

            except Exception as e:
                print(f"Erro ao extrair {p.name}: {e}")

        elif p.suffix.lower() == ".csv":
            direct_dir = EXTRACT_DIR / "direct_csv"
            direct_dir.mkdir(parents=True, exist_ok=True)
            shutil.copy2(p, direct_dir / p.name)

    print("\nUpload e extração concluídos.")


# ==============================================================================
# FUNÇÕES AUXILIARES
# ==============================================================================

def normalize_text(s: str) -> str:
    s = str(s)
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = s.lower()
    s = s.replace("-", "_")
    s = s.replace(" ", "_")
    s = re.sub(r"_+", "_", s)
    return s


def slugify(s: str) -> str:
    s = str(s).strip()
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"[^A-Za-z0-9_\-]+", "", s)
    return s[:140] if s else "UnknownDataset"


def is_raw_results_file(path: Path) -> bool:
    """
    Mantém somente arquivos de resultados brutos.
    Ignora resumos, relatórios, rankings e arquivos agregados.
    """
    name = normalize_text(path.name)

    is_raw = (
        "resultados_brutos" in name
        or "raw_results" in name
        or name == "raw_results.csv"
    )

    is_not_raw = (
        "resumo" in name
        or "summary" in name
        or "stats" in name
        or "estatisticas" in name
        or "relatorio" in name
        or "report" in name
        or "por_estrategia" in name
        or "ranking" in name
        or "consolidated" in name
        or "consolidado" in name
        or "stagewise" in name
        or "diagnostic" in name
        or "diagnostico" in name
        or "all_" in name
    )

    return is_raw and not is_not_raw


def detect_attack_from_text(text: str):
    """
    Detecta o ataque usando o caminho completo.
    """
    s = normalize_text(text)

    if (
        "cumulativo" in s
        or "cumulative" in s
        or "asr_fixo" in s
        or "asrfixo" in s
        or "slowdrift" in s
        or "slow_drift" in s
        or "drift" in s
    ):
        return "Cumulative"

    if (
        "modelreplacement" in s
        or "model_replacement" in s
        or "scaling" in s
        or "escala" in s
        or "escalonamento" in s
    ):
        return "Scaling"

    if (
        "sybil" in s
        or "identityflood" in s
        or "identity_flood" in s
    ):
        return "Sybil"

    if (
        "onoff" in s
        or "on_off" in s
        or "sleepingcell" in s
        or "sleeping_cell" in s
        or "sleeping" in s
    ):
        return "OnOff"

    return None


def detect_dataset_from_text(text: str):
    """
    Padroniza o nome do dataset.
    """
    original = str(text)
    s = normalize_text(original)

    if (
        "dataset_a" in s
        or "cps_netflow" in s
        or "cps_netflow_ids" in s
        or "cps-netflow" in original.lower()
    ):
        return "A_CPS-NetFlow-IDS"

    if (
        "dataset_b" in s
        or "packet_level" in s
        or "packet-level" in original.lower()
        or "botnet" in s
    ):
        return "B_Packet-Level Botnet Set"

    if (
        "dataset_c" in s
        or "mqtt_iot" in s
        or "mqtt-iot" in original.lower()
        or "mqtt" in s
    ):
        return "C_MQTT-IoT-Flood"

    if (
        "dataset_d" in s
        or "aggregated_traffic" in s
        or "aggregated traffic" in original.lower()
    ):
        return "D_Aggregated Traffic Set"

    return "UnknownDataset"


def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Padroniza possíveis variações nos nomes de colunas.
    """
    rename = {}

    for c in df.columns:
        cl = normalize_text(c)

        if cl in ["seed", "semente"]:
            rename[c] = "Seed"

        elif cl in ["strategy", "estrategia"]:
            rename[c] = "Strategy"

        elif cl in ["stage", "estagio", "estagio_idx"]:
            rename[c] = "Stage"

        elif cl in ["stage_name", "nome_stage"]:
            rename[c] = "Stage_Name"

        elif cl in ["f1", "f1_score", "f1score"]:
            rename[c] = "F1"

        elif cl in ["asr", "attack_success_rate"]:
            rename[c] = "ASR"

        elif cl in [
            "time",
            "tempo",
            "runtime",
            "execution_time",
            "stage_time",
            "time_seconds",
            "tempo_segundos"
        ]:
            rename[c] = "Time"

        elif cl in [
            "cumulative_time_strategy",
            "cumtime",
            "cum_time",
            "tempo_acumulado"
        ]:
            rename[c] = "Cumulative_Time_Strategy"

        elif cl in ["benchmark"]:
            rename[c] = "Benchmark"

        elif cl in ["attack", "ataque"]:
            rename[c] = "Attack"

        elif cl in ["dataset", "base"]:
            rename[c] = "Dataset"

    return df.rename(columns=rename)


# ==============================================================================
# CARREGAR RESULTADOS BRUTOS
# ==============================================================================

def load_raw_results():
    all_csvs = sorted(set([p.resolve() for p in EXTRACT_DIR.rglob("*.csv")]))

    print("\n" + "=" * 90)
    print("BUSCA DE CSVs")
    print("=" * 90)
    print(f"CSV encontrados: {len(all_csvs)}")

    if not all_csvs:
        raise RuntimeError("Nenhum arquivo CSV encontrado nos dados enviados.")

    raw_csvs = [p for p in all_csvs if is_raw_results_file(p)]

    print(f"CSV brutos reconhecidos: {len(raw_csvs)}")

    if not raw_csvs:
        print("\nArquivos CSV encontrados, mas nenhum reconhecido como resultado bruto:")
        for p in all_csvs[:80]:
            print(" -", p)
        raise RuntimeError("Nenhum arquivo bruto encontrado. Verifique nomes contendo Resultados_Brutos ou Raw_Results.")

    REQUIRED_BASE = {"Seed", "Strategy", "Stage", "F1", "ASR"}

    frames = []
    skipped = []

    for p in raw_csvs:
        try:
            df = pd.read_csv(p)
        except Exception as e:
            skipped.append((p, f"erro de leitura: {e}"))
            continue

        df = standardize_columns(df)

        missing = REQUIRED_BASE - set(df.columns)

        if missing:
            skipped.append((p, f"colunas obrigatórias ausentes: {missing}; colunas={list(df.columns)}"))
            continue

        if "Time" not in df.columns:
            df["Time"] = pd.NA

        # Detecta ataque
        if "Attack" in df.columns and df["Attack"].notna().any():
            attack = detect_attack_from_text(df["Attack"].dropna().iloc[0])
            if attack is None:
                attack = detect_attack_from_text(str(p))
        else:
            attack = detect_attack_from_text(str(p))

        if attack is None:
            skipped.append((p, "ataque não detectado pelo caminho completo"))
            continue

        # Detecta dataset
        if "Dataset" in df.columns and df["Dataset"].notna().any():
            dataset_candidate = str(df["Dataset"].dropna().iloc[0])
            dataset = detect_dataset_from_text(dataset_candidate)

            if dataset == "UnknownDataset":
                dataset = detect_dataset_from_text(str(p))
        else:
            dataset = detect_dataset_from_text(str(p))

        if dataset == "UnknownDataset":
            skipped.append((p, "dataset não detectado"))
            continue

        keep_cols = ["Seed", "Strategy", "Stage", "F1", "ASR", "Time"]

        if "Cumulative_Time_Strategy" in df.columns:
            keep_cols.append("Cumulative_Time_Strategy")

        df = df[keep_cols].copy()

        for col in ["Seed", "Stage", "F1", "ASR", "Time"]:
            df[col] = pd.to_numeric(df[col], errors="coerce")

        if "Cumulative_Time_Strategy" in df.columns:
            df["Cumulative_Time_Strategy"] = pd.to_numeric(
                df["Cumulative_Time_Strategy"],
                errors="coerce"
            )

        df = df.dropna(subset=["Seed", "Strategy", "Stage", "F1", "ASR"])

        df["Attack"] = attack
        df["Dataset"] = dataset
        df["SourceFile"] = p.name
        df["SourcePath"] = str(p)

        frames.append(df)

    print(f"CSV brutos carregados: {len(frames)}")
    print(f"CSV ignorados: {len(skipped)}")

    if skipped:
        print("\nArquivos ignorados:")
        for p, reason in skipped[:50]:
            print(" -", p.name, "=>", reason)

    if not frames:
        raise RuntimeError("Nenhum CSV válido foi carregado.")

    df_all = pd.concat(frames, ignore_index=True)

    print("\nAtaques detectados:")
    print(sorted(df_all["Attack"].unique()))

    print("\nDatasets detectados:")
    print(sorted(df_all["Dataset"].unique()))

    print("\nCombinações Attack x Dataset:")
    print(
        df_all[["Attack", "Dataset"]]
        .drop_duplicates()
        .sort_values(["Attack", "Dataset"])
        .to_string(index=False)
    )

    print("\nLinhas carregadas:", len(df_all))

    return df_all


# ==============================================================================
# AGREGAÇÃO
# ==============================================================================

def build_stagewise_statistics(df_all: pd.DataFrame):
    agg_dict = {
        "F1": ["mean", "std"],
        "ASR": ["mean", "std"]
    }

    if df_all["Time"].notna().any():
        agg_dict["Time"] = ["mean", "std"]

    if "Cumulative_Time_Strategy" in df_all.columns:
        if df_all["Cumulative_Time_Strategy"].notna().any():
            agg_dict["Cumulative_Time_Strategy"] = ["mean", "std"]

    df_stats = (
        df_all
        .groupby(["Attack", "Dataset", "Strategy", "Stage"], as_index=False)
        .agg(agg_dict)
    )

    df_stats.columns = [
        "_".join([c for c in col if c]).strip("_")
        if isinstance(col, tuple)
        else col
        for col in df_stats.columns
    ]

    for col in [
        "F1_std",
        "ASR_std",
        "Time_std",
        "Cumulative_Time_Strategy_std"
    ]:
        if col in df_stats.columns:
            df_stats[col] = df_stats[col].fillna(0)

    return df_stats


# ==============================================================================
# PLOTS
# ==============================================================================

def plot_stagewise(df_bd, attack, dataset, metric_mean, metric_std, ylabel, outpath: Path):
    df_bd = df_bd.sort_values(["Stage", "Strategy"])

    fig, ax = plt.subplots(figsize=(10, 4.8), dpi=180)

    for strat, g in df_bd.groupby("Strategy"):
        g = g.sort_values("Stage")

        x = g["Stage"].values
        y = g[metric_mean].values

        ax.plot(
            x,
            y,
            marker="o",
            linewidth=1.8,
            markersize=3.0,
            label=strat
        )

        if metric_std in g.columns:
            std = g[metric_std].fillna(0).values

            ax.fill_between(
                x,
                y - std,
                y + std,
                alpha=0.12
            )

    ax.set_title(
        f"{attack} — {dataset}\nMean {ylabel} over stages",
        fontsize=12
    )

    ax.set_xlabel("Stage")
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.25)
    ax.set_xticks(sorted(df_bd["Stage"].unique()))

    if ylabel.upper() in ["ASR", "F1-SCORE"]:
        ax.set_ylim(-0.02, 1.02)

    ax.legend(
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        frameon=False,
        fontsize=9
    )

    outpath.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(outpath, bbox_inches="tight")
    plt.close(fig)


def generate_plots(df_stats: pd.DataFrame):
    generated = []

    for attack in sorted(df_stats["Attack"].unique()):
        df_a = df_stats[df_stats["Attack"] == attack]

        for dataset in sorted(df_a["Dataset"].unique()):
            df_ad = df_a[df_a["Dataset"] == dataset]

            ds_slug = slugify(dataset)

            f1_out = OUT_DIR / attack / ds_slug / f"{attack}_{ds_slug}_F1_over_stage.png"
            asr_out = OUT_DIR / attack / ds_slug / f"{attack}_{ds_slug}_ASR_over_stage.png"

            plot_stagewise(
                df_ad,
                attack,
                dataset,
                "F1_mean",
                "F1_std",
                "F1-score",
                f1_out
            )

            plot_stagewise(
                df_ad,
                attack,
                dataset,
                "ASR_mean",
                "ASR_std",
                "ASR",
                asr_out
            )

            generated += [f1_out, asr_out]

            if GENERATE_TIME_PLOTS:
                if "Time_mean" in df_ad.columns and df_ad["Time_mean"].notna().any():
                    time_out = OUT_DIR / attack / ds_slug / f"{attack}_{ds_slug}_Time_over_stage.png"

                    plot_stagewise(
                        df_ad,
                        attack,
                        dataset,
                        "Time_mean",
                        "Time_std",
                        "Time (seconds)",
                        time_out
                    )

                    generated.append(time_out)

            if "Cumulative_Time_Strategy_mean" in df_ad.columns:
                if df_ad["Cumulative_Time_Strategy_mean"].notna().any():
                    cumtime_out = OUT_DIR / attack / ds_slug / f"{attack}_{ds_slug}_CumulativeTime_over_stage.png"

                    plot_stagewise(
                        df_ad,
                        attack,
                        dataset,
                        "Cumulative_Time_Strategy_mean",
                        "Cumulative_Time_Strategy_std",
                        "Cumulative Time (seconds)",
                        cumtime_out
                    )

                    generated.append(cumtime_out)

    print(f"\nFiguras geradas: {len(generated)}")
    print(f"Diretório: {OUT_DIR}")

    return generated


# ==============================================================================
# RANKING FINAL E DIAGNÓSTICO
# ==============================================================================

def build_final_ranking(df_stats: pd.DataFrame):
    final_rows = []

    for (attack, dataset, strategy), g in df_stats.groupby(["Attack", "Dataset", "Strategy"]):
        max_stage = g["Stage"].max()
        final_rows.append(g[g["Stage"] == max_stage])

    df_final = pd.concat(final_rows, ignore_index=True)

    df_final = df_final.sort_values(
        ["Attack", "Dataset", "ASR_mean", "F1_mean"],
        ascending=[True, True, True, False]
    )

    return df_final


def build_diagnostic(df_all: pd.DataFrame):
    diag = (
        df_all
        .groupby(["Attack", "Dataset"])
        .agg(
            rows=("F1", "count"),
            min_stage=("Stage", "min"),
            max_stage=("Stage", "max"),
            strategies=("Strategy", "nunique"),
            seeds=("Seed", "nunique"),
            has_time=("Time", lambda x: x.notna().any())
        )
        .reset_index()
        .sort_values(["Attack", "Dataset"])
    )

    return diag


# ==============================================================================
# ZIP
# ==============================================================================

def create_output_zip(generated_files):
    zip_path = Path("/content") / ZIP_OUTPUT_NAME

    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        # Figuras
        for p in generated_files:
            if Path(p).exists():
                z.write(p, arcname=Path(p).relative_to(WORK_DIR).as_posix())

        # CSVs
        for fname in [
            "benchmark_all_raw_consolidated.csv",
            "benchmark_stagewise_stats.csv",
            "benchmark_final_stage_ranking.csv",
            "benchmark_diagnostic_by_attack_dataset.csv"
        ]:
            p = WORK_DIR / fname
            if p.exists():
                z.write(p, arcname=p.name)

    return zip_path


# ==============================================================================
# MAIN
# ==============================================================================

def main():
    reset_workspace()

    upload_and_extract_files()

    df_all = load_raw_results()

    df_stats = build_stagewise_statistics(df_all)

    df_final = build_final_ranking(df_stats)

    diag = build_diagnostic(df_all)

    # Salvar CSVs no WORK_DIR
    df_all.to_csv(WORK_DIR / "benchmark_all_raw_consolidated.csv", index=False)
    df_stats.to_csv(WORK_DIR / "benchmark_stagewise_stats.csv", index=False)
    df_final.to_csv(WORK_DIR / "benchmark_final_stage_ranking.csv", index=False)
    diag.to_csv(WORK_DIR / "benchmark_diagnostic_by_attack_dataset.csv", index=False)

    print("\nRanking final preview:")
    display(df_final.head(40))

    print("\nDiagnóstico por ataque/dataset:")
    display(diag)

    generated_files = generate_plots(df_stats)

    zip_path = create_output_zip(generated_files)

    print("\n" + "=" * 90)
    print("ARQUIVO FINAL GERADO")
    print("=" * 90)
    print(f"ZIP: {zip_path}")
    print("=" * 90)

    files.download(zip_path.as_posix())


if __name__ == "__main__":
    main()

UPLOAD DOS RESULTADOS DO BENCHMARK
Faça upload de arquivos .zip ou .csv.
Você pode enviar vários arquivos de uma vez.


Saving RESULTADOS_CONSOLIDADOS_FINAIS.zip to RESULTADOS_CONSOLIDADOS_FINAIS (1).zip

Arquivos enviados:
 - RESULTADOS_CONSOLIDADOS_FINAIS (1).zip

Extraindo arquivos ZIP...
Extraído: RESULTADOS_CONSOLIDADOS_FINAIS (1).zip -> /content/stagewise_graphs/extracted/RESULTADOS_CONSOLIDADOS_FINAIS (1)

Upload e extração concluídos.

BUSCA DE CSVs
CSV encontrados: 85
CSV brutos reconhecidos: 17
CSV brutos carregados: 17
CSV ignorados: 0

Ataques detectados:
['Cumulative', 'OnOff', 'Scaling', 'Sybil']

Datasets detectados:
['A_CPS-NetFlow-IDS', 'B_Packet-Level Botnet Set', 'C_MQTT-IoT-Flood', 'D_Aggregated Traffic Set']

Combinações Attack x Dataset:
    Attack                   Dataset
Cumulative         A_CPS-NetFlow-IDS
Cumulative B_Packet-Level Botnet Set
Cumulative          C_MQTT-IoT-Flood
Cumulative  D_Aggregated Traffic Set
     OnOff         A_CPS-NetFlow-IDS
     OnOff B_Packet-Level Botnet Set
     OnOff          C_MQTT-IoT-Flood
     OnOff  D_Aggregated Traffic Set
   Scaling       

,Attack,Dataset,Strategy,Stage,F1_mean,F1_std,ASR_mean,ASR_std,Time_mean,Time_std,Cumulative_Time_Strategy_mean,Cumulative_Time_Strategy_std
0,Cumulative,A_CPS-NetFlow-IDS,Adaptive_Ultimate,16,0.690321,0.218877,0.619681,0.428586,0.075428,0.039931,1.346322,0.842573
4,Cumulative,A_CPS-NetFlow-IDS,Krum,16,0.687439,0.242659,0.633395,0.424915,0.076360,0.044703,1.385929,0.888349
2,Cumulative,A_CPS-NetFlow-IDS,FLTrust,16,0.705775,0.224320,0.636103,0.431998,0.073563,0.039242,1.343988,0.935062
1,Cumulative,A_CPS-NetFlow-IDS,FLAME,16,0.687285,0.240224,0.674624,0.401859,0.072294,0.039469,1.304689,0.807636
3,Cumulative,A_CPS-NetFlow-IDS,FedAvg,16,0.692326,0.239156,0.696850,0.402812,0.084290,0.084765,1.328981,0.727036
9,Cumulative,B_Packet-Level Botnet Set,Krum,16,0.631777,0.176527,0.340000,0.571664,0.047299,0.016883,0.821381,0.227320
6,Cumulative,B_Packet-Level Botnet Set,FLAME,16,0.684481,0.085825,0.666667,0.577350,0.042268,0.004073,0.747175,0.079177
8,Cumulative,B_Packet-Level Botnet Set,FedAvg,16,0.684481,0.085825,0.673333,0.565803,0.046562,0.013494,0.797585,0.195686
5,Cumulative,B_Packet-Level Botnet Set,Adaptive_Ultimate,16,0.682635,0.083836,0.673333,0.565803,0.046261,0.012367,0.736760,0.086639
7,Cumulative,B_Packet-Level Botnet Set,FLTrust,16,0.682635,0.083836,0.673333,0.565803,0.041890,0.005725,0.709213,0.036760



Diagnóstico por ataque/dataset:


,Attack,Dataset,rows,min_stage,max_stage,strategies,seeds,has_time
0,Cumulative,A_CPS-NetFlow-IDS,4335,0,16,5,3,True
1,Cumulative,B_Packet-Level Botnet Set,255,0,16,5,3,True
2,Cumulative,C_MQTT-IoT-Flood,255,0,16,5,3,True
3,Cumulative,D_Aggregated Traffic Set,255,0,16,5,3,True
4,OnOff,A_CPS-NetFlow-IDS,255,0,16,5,3,True
5,OnOff,B_Packet-Level Botnet Set,255,0,16,5,3,True
6,OnOff,C_MQTT-IoT-Flood,255,0,16,5,3,True
7,OnOff,D_Aggregated Traffic Set,255,0,16,5,3,True
8,Scaling,A_CPS-NetFlow-IDS,255,0,16,5,3,True
9,Scaling,B_Packet-Level Botnet Set,255,0,16,5,3,True



Figuras geradas: 64
Diretório: /content/stagewise_graphs/figs_by_dataset

ARQUIVO FINAL GERADO
ZIP: /content/figs_stagewise_by_dataset_ALL_attacks_FIXED_FULL.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>